# NLP Financial Signals — Exploratory Data Analysis

This notebook explores the sample central bank communications dataset,
examines topic distributions, sentiment patterns, and temporal trends
across the Bank of England, Federal Reserve, and ECB.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 100)

print('Setup complete')

## 1. Load Sample Data

In [ ]:
speeches = pd.read_csv('../data/sample/sample_speeches.csv', parse_dates=['date'])
topics = pd.read_csv('../data/sample/sample_topics.csv')
sentiment = pd.read_csv('../data/sample/sample_sentiment.csv')

print(f'Speeches: {len(speeches)} documents')
print(f'Topics: {len(topics)} assignments')
print(f'Sentiment: {len(sentiment)} scores')
speeches.head()

## 2. Document Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# By source
speeches['source'].value_counts().plot(kind='bar', ax=axes[0], color=['#0072C6', '#003366', '#003399'])
axes[0].set_title('Documents by Source')
axes[0].set_ylabel('Count')

# By type
speeches['doc_type'].value_counts().plot(kind='bar', ax=axes[1], color='#2196F3')
axes[1].set_title('Documents by Type')

# Over time
speeches.set_index('date').resample('Q').size().plot(ax=axes[2], color='#FF5722')
axes[2].set_title('Documents Over Time (Quarterly)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Topic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Topic counts
topic_counts = topics['topic_name'].value_counts()
topic_counts.plot(kind='barh', ax=axes[0], color='#4CAF50')
axes[0].set_title('Document Count by Topic')
axes[0].set_xlabel('Count')

# Topic by source
merged = speeches.merge(topics, left_index=True, right_on='doc_index')
ct = pd.crosstab(merged['source'], merged['topic_name'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Topic Distribution by Central Bank (%)')
axes[1].set_ylabel('Percentage')
axes[1].legend(bbox_to_anchor=(1.05, 1), fontsize=8)

plt.tight_layout()
plt.show()

## 4. Sentiment Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Compound distribution
axes[0].hist(sentiment['sentiment_compound'], bins=30, color='#2196F3', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Compound Sentiment Distribution')
axes[0].set_xlabel('Compound Score')

# Labels
sentiment['sentiment_label'].value_counts().plot(
    kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#4CAF50', '#FF5722', '#607D8B']
)
axes[1].set_title('Sentiment Label Distribution')
axes[1].set_ylabel('')

# By source
for source in ['boe', 'fed', 'ecb']:
    subset = sentiment[sentiment['source'] == source]
    axes[2].hist(subset['sentiment_compound'], bins=15, alpha=0.5, label=source.upper())
axes[2].legend()
axes[2].set_title('Sentiment by Central Bank')

plt.tight_layout()
plt.show()

## 5. Temporal Trends

In [ ]:
sent_dated = sentiment.copy()
sent_dated['date'] = pd.to_datetime(sent_dated['date'])
sent_dated['year_month'] = sent_dated['date'].dt.to_period('M')

fig, ax = plt.subplots(figsize=(14, 5))

colors = {'boe': '#0072C6', 'fed': '#003366', 'ecb': '#003399'}
for source in ['boe', 'fed', 'ecb']:
    subset = sent_dated[sent_dated['source'] == source]
    monthly = subset.groupby('year_month')['sentiment_compound'].mean()
    smoothed = monthly.rolling(3, min_periods=1).mean()
    ax.plot(range(len(smoothed)), smoothed.values,
            label=source.upper(), color=colors[source], linewidth=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax.set_title('Sentiment Trends by Central Bank (3-month rolling)', fontsize=14)
ax.set_ylabel('Mean Compound Sentiment')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Word Count Analysis

In [ ]:
speeches['word_count'] = speeches['text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(10, 5))
for source in ['boe', 'fed', 'ecb']:
    subset = speeches[speeches['source'] == source]
    ax.hist(subset['word_count'], bins=20, alpha=0.5,
            label=source.upper(), color=colors[source])

ax.set_title('Document Length Distribution by Source')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

print(speeches.groupby('source')['word_count'].describe().round(0))